# 1.Compute total cost per order in new column 
# 2.Create another column and Name that column as premium_customer. Those customer whose quantity is greater than 15 are premium_customer

In [0]:
%pip install openpyxl
import pandas as pd

# Step 1: Read Excel into pandas
pdf = pd.read_excel("/Volumes/workspace/sales/sales/solution/sales.xlsx")

# Step 2: Convert pandas to PySpark
df = spark.createDataFrame(pdf)

display(df)

In [0]:
# 1.Compute total cost per order in new column
from pyspark.sql.functions import col,round
df_total_cost_per_order=df.withColumn("Total_cost",round(col("Quantity")*col("Price"),2))
display(df_total_cost_per_order)


In [0]:
# 2.Create another column and Name that column as premium_customer. Those customer whose quantity is greater than 15 are premium_customer
from pyspark.sql.functions import sum,col
df_group_by_cutomer_with_quantity=df.groupBy("Customer").agg(sum(col("Quantity").cast("int")).alias("total_quantity_per_customer"))
display(df_group_by_cutomer_with_quantity)

In [0]:
from pyspark.sql.functions import when
premium_customer_df=df_group_by_cutomer_with_quantity.withColumn("premium_customer",when(col("total_quantity_per_customer")>15,"True").otherwise("False"))
display(premium_customer_df)

In [0]:
join_df=df.join(premium_customer_df,df.Customer==premium_customer_df.Customer,"inner").drop(premium_customer_df.Customer)
display(join_df)

In [0]:
# df2 = df.replace({"USA": "United States", "UK": "United Kingdom"}, subset=["country"])

# df = (df
#       .withColumn("age", col("age") + 1)
#       .withColumn("country", when(col("country") == "USA", "United States"))
#      )


# df = df.withColumn(
#     "age",
#     when(col("name") == "Rohit", 30).otherwise(col("age"))
# )


df1=df
df2=df.withColumn("Price",when(col("OrderID")==5,60).otherwise(col("Price"))).withColumn("Quantity",when(col("OrderID")==1,80).otherwise(col("Quantity")))
display(df2)


# orderId=5 -> price= 60
# orderid=1 -> quantity=80

In [0]:
# load both df1 and df2 in new folder volume
from datetime import datetime
today_str = datetime.today().strftime("%Y-%m-%d")

path_df1=f"/Volumes/workspace/sales/sales/df1/{today_str}.csv"
path_df2=f"/Volumes/workspace/sales/sales/df2/{today_str}.csv"
df1.write.format("csv").mode("append").option("header","True").save(path_df1)
df2.write.format("csv").mode("append").option("header","True").save(path_df2)

SCD1

In [0]:
#  Table as Target

create_table_script = """
CREATE TABLE IF NOT EXISTS sales.df1_target_scd1_table (
  OrderID LONG NOT NULL,
  Item STRING NOT NULL,
  Quantity LONG NOT NULL,
  Price DOUBLE NOT NULL,
  Customer STRING
)
"""
spark.sql(create_table_script)

In [0]:

df1.write.mode("append").option("overwriteSchema","true").saveAsTable("sales.df1_target_scd1_table")

In [0]:
%sql

select * from sales.df1_target_scd1_table

In [0]:
#  Table as Source

create_table_script = """
CREATE TABLE IF NOT EXISTS sales.df2_source_scd1_table (
  OrderID LONG NOT NULL,
  Item STRING NOT NULL,
  Quantity LONG NOT NULL,
  Price DOUBLE NOT NULL,
  Customer STRING
)
"""
spark.sql(create_table_script)

In [0]:
df2.write.mode("append").option("overwriteSchema","true").saveAsTable("sales.df2_source_scd1_table")

In [0]:
%sql
select * from sales.df2_source_scd1_table

In [0]:
%sql


WITH SourceDeduped AS (
  SELECT
    OrderID,
    FIRST(Item) AS Item,
    FIRST(Quantity) AS Quantity,
    FIRST(Price) AS Price,
    FIRST(Customer) AS Customer
  FROM sales.df2_source_scd1_table
  GROUP BY OrderID
)
MERGE INTO sales.df1_target_scd1_table AS Target
USING SourceDeduped AS Source
ON Target.OrderID = Source.OrderID
WHEN MATCHED THEN
  UPDATE SET
    Target.Quantity = Source.Quantity,
    Target.Price = Source.Price
WHEN NOT MATCHED BY Target THEN
  INSERT (OrderId, Item, Quantity, Price, Customer)
  VALUES (Source.OrderId, Source.Item, Source.Quantity, Source.Price, Source.Customer);



-- Purpose: Deduplicate the source data.

-- If multiple rows have the same OrderID, you pick the FIRST value for each column.

-- GROUP BY OrderID → ensures one record per OrderID.

-- This prepares a clean, deduped source dataset before the MERGE.




-- MERGE INTO sales.df1_target_scd1_table AS Target
-- USING sales.df2_source_scd1_table AS Source
-- ON Target.OrderID=Source.OrderID
-- WHEN MATCHED THEN
--   UPDATE SET Target.Quantity=Source.Quantity,
--              Target.Price=Source.Price
-- WHEN NOT MATCHED BY Target THEN
--   INSERT (OrderId,Item,Quantity,Price,Customer)
--   VALUES(Source.OrderId, Source.Item,Source.Quantity,Source.Price,Source.Customer);
             


-- MERGE INTO Products AS T
-- USING Products_Stage AS S
-- ON T.ProductKey = S.ProductKey
-- WHEN MATCHED THEN
-- UPDATE SET T.StockLevel = S.StockLevel,
-- T.Price = S.Price
-- WHEN NOT MATCHED BY T THEN
-- INSERT (ProductKey, ProductName, StockLevel, Price)
-- VALUES (S.ProductKey, S.ProductName, S.StockLevel, S.Price);


In [0]:
%sql
select * from sales.df1_target_scd1_table

In [0]:
%sql
select * from sales.df2_source_scd1_table

SCD2

In [0]:
from datetime import datetime, timedelta

# start_date_default = (datetime.today() + timedelta(days=10)).strftime("%Y-%m-%d")

create_table_script = f"""
CREATE TABLE IF NOT EXISTS sales.df1_target_scd2_table (
  OrderID LONG NOT NULL,
  Item STRING NOT NULL,
  Quantity LONG NOT NULL,
  Price DOUBLE NOT NULL,
  Customer STRING,
  StartDate DATE,
  EndDate DATE,
  isActive INT 
)
"""
spark.sql(create_table_script)

In [0]:
from pyspark.sql.functions import lit
from datetime import datetime, timedelta

start_date_default = (datetime.today() - timedelta(days=10)).strftime("%Y-%m-%d")

df1_target_scd2 = df1.withColumn("StartDate", lit(start_date_default)) \
    .withColumn("EndDate", lit(None).cast("date")) \
    .withColumn("isActive", lit(1))

display(df1_target_scd2)

In [0]:
df1_target_scd2=df1_target_scd2.withColumn("StartDate", col("StartDate").cast("date"))

df1_target_scd2.write.mode("append").option("overwriteSchema","true").saveAsTable("sales.df1_target_scd2_table")
display(df1_target_scd2)

In [0]:
create_table_script = """
CREATE TABLE IF NOT EXISTS sales.df2_source_scd2_table (
  OrderID LONG NOT NULL,
  Item STRING NOT NULL,
  Quantity LONG NOT NULL,
  Price DOUBLE NOT NULL,
  Customer STRING
)
"""
spark.sql(create_table_script)

In [0]:
from pyspark.sql import Row

new_rows = [
    Row(OrderID=21, Item="Item21", Quantity=24, Price=30.2,Customer="Customer_3"),
    Row(OrderID=22, Item="Item22", Quantity=4, Price=3.6,Customer="Customer_4"),
    Row(OrderID=23, Item="Item23", Quantity=44, Price=3.5,Customer="Customer_5"),
]

df_new = spark.createDataFrame(new_rows)

final_df2_source_scd2 = df2.unionByName(df_new)

display(final_df2_source_scd2)

In [0]:
final_df2_source_scd2.write.mode("append").option("overwriteSchema","true").saveAsTable("sales.df2_source_scd2_table")

In [0]:
%sql


-- OrderID:long
-- Item:string
-- Quantity:long
-- Price:double
-- Customer:string
-- StartDate:date
-- EndDate:date
-- isActive:integer

-- df1_target_scd2
-- today_str = datetime.today().strftime("%Y-%m-%d")



MERGE INTO sales.df1_target_scd2_table  AS Target
USING sales.df2_source_scd2_table AS Source
ON Target.OrderID = Source.OrderID AND Target.IsActive = 1
WHEN MATCHED THEN
  UPDATE SET
    Target.IsActive = 0,
    Target.EndDate = date_add(DAY, -1, current_date())
WHEN NOT MATCHED THEN
  INSERT (
    OrderID, Item, Quantity, Price, Customer, StartDate, EndDate, IsActive
  )
  VALUES (
    Source.OrderID, Source.Item, Source.Quantity, Source.Price, Source.Customer, current_date(), NULL, 1
  )






-- INSERT INTO df1_target_scd2 (OrderID, Item, Quantity, Price,Customer, StartDate, EndDate, IsActive)
--     SELECT 
--         OrderID,
--         Item,
--         Quantity,
--         Price,
--         Customer,
--         GETDATE() AS StartDate,
--         NULL AS EndDate,
--         1 AS IsActive
--     FROM (
--         MERGE df1_target_scd2 AS Target
--         USING final_df2_source_scd2 AS Source
--         ON Target.df1_target_scd2 = Source.final_df2_source_scd2
--         WHEN MATCHED AND Target.IsActive = 1 THEN
--             -- Deactivate existing row
--             UPDATE SET T.IsActive = 0, T.EndDate = DATEADD(DAY, -1, GETDATE())
--         WHEN NOT MATCHED BY TARGET THEN
--             -- Insert new rows (handled in outer SELECT)
--             INSERT (OrderId, Item, Quantity, Price, Customer,StartDate,EndDate,IsActive)
--               VALUES (Source.OrderId, Source.Item, Source.Quantity, Source.Price, Source.Customer,current_date(),NULL,1)
--     );
            -- NOTHING
        -- OUTPUT 
        --     S.ProductKey,
        --     S.ProductName,
        --     S.StockLevel,
        --     S.Price,
        --     $action AS Operation
    -- ) AS MergeOutput
    -- WHERE MergeOutput.Operation = 'UPDATE'
    -- OR MergeOutput.Operation = 'INSERT';  -- capture both new rows and updates

In [0]:
%sql
SELECT * FROM sales.df1_target_scd2_table 